In [36]:
%reload_ext autoreload
%autoreload 2

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [38]:
from solarrpy import seasonalClearsky
from solarrpy import seasonalModel

In [39]:
"""
import cdsapi

dataset = "cams-solar-radiation-timeseries"
request = {
    "sky_type": "observed_cloud",
    "location": {"longitude": 11.3426, "latitude": 44.4949},
    "altitude": ["71"],
    "date": ["2005-01-01/2026-03-31"],
    "time_step": "1day",
    "time_reference": "true_solar_time",
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()
"""

'\nimport cdsapi\n\ndataset = "cams-solar-radiation-timeseries"\nrequest = {\n    "sky_type": "observed_cloud",\n    "location": {"longitude": 11.3426, "latitude": 44.4949},\n    "altitude": ["71"],\n    "date": ["2005-01-01/2026-03-31"],\n    "time_step": "1day",\n    "time_reference": "true_solar_time",\n    "data_format": "csv"\n}\n\nclient = cdsapi.Client()\nclient.retrieve(dataset, request).download()\n'

In [40]:
df = pd.read_csv("../data/Bologna.csv")
spec = {
    'target': 'GHI',
    'coords': {
        "lat": 44.4949,
        "lon": 11.3426,
        "alt": 71
    },
    'data': df
}

In [41]:
# ----------------------------------------------------------------------------
# NOTE: This script assumes the existence of the previously translated classes/functions:
# SeasonalClearsky, control_seasonalClearsky, clearsky_outliers
# 
# It also assumes a pre-existing `spec` object/dictionary that contains the dataset.
# For example:
# spec = {
#     'data': pd.DataFrame({'GHI': [...], 'date': [...], 'clearsky': [...], 'n': [...]}),
#     'coords': {'lat': 45.0},
#     'target': 'GHI'
# }
# ----------------------------------------------------------------------------

# ==========================================
# Inputs
# ==========================================

# Control parameters
control = seasonalClearsky.control_seasonalClearsky(
    orders=1, order_H0=1, periods=365, 
    include_intercept=True, include_trend=False,
    delta0=1.4, lower=0, upper=3, by=0.001, ntol=0, quiet=False
)

# Extracting from `spec` object
data_all = spec['data'].copy()
data_all['date'] = pd.to_datetime(data_all['date'])
lat = spec['coords']['lat']
alt = spec['coords']['alt']
target_col = spec['target']

# Command parameters
plot_data = False
test_outputs = False

all_params = []
computed_values = []

for year_i in range(2013, 2023):
    mask = data_all['date'].dt.year <= year_i
    data = data_all.loc[mask].copy()
    
    model_coefficients = {}

    print(f"\nRunning model with data up to {year_i}-12-31 ({len(data)} rows)")

    GHI = data['GHI']
    date = data['date']
    clearsky = data['clearsky']
    H0 = data['H0']

    # ==========================================
    # Fit the clear sky model
    # ==========================================

    # Initialize the model 
    clearsky_model = seasonalClearsky.SeasonalClearsky(control=control)

    # Fit the parameters
    clearsky_model.fit(GHI, date, lat, clearsky, alt=alt)
    #print(clearsky_model)
    
    # Predictions
    Ct = clearsky_model.predict(n=data['n'], newdata=data)
    data['Ct'] = Ct
    #print(f"Ct: [{min(data['Ct']):.2f},{max(data['Ct']):.2f}], GHI: [{min(data['GHI']):.2f},{max(data['GHI']):.2f}]")
    #df = pd.DataFrame(data[['n', 'GHI', 'Ct']])
    #display(df.iloc[np.where(df['GHI'] >= df['Ct'])])

    # Save the model coefficients
    delta0, delta1, delta2, delta3 = clearsky_model._model.params.values[:4]
    delta0_err, delta1_err, delta2_err, delta3_err = clearsky_model._model.bse.values[:4]

    #Computing alpha and beta
    ratio = GHI / Ct
    eps = 1e-3 * min(1 - ratio)

    alpha_t = np.min(1 - ratio) - eps
    beta_t  = max(1 - ratio) - min(1 - ratio) + 2 * eps

    # Handle edge cases: ensure valid domain for logarithms
    # 1 - α - R_t/C_t must be > 0 and < β
    inner_arg = 1 - alpha_t - ratio

    # Clip to valid range
    #eps = 1e-6
    inner_arg = np.clip(inner_arg, eps, beta_t - eps)

    # Apply double logarithm transformation: Y_t = log(log(β) - log(1 - α - R_t/C_t))
    log_inner = np.log(inner_arg)
    log_beta = np.log(beta_t)
    outer_arg = log_beta - log_inner

    # Ensure outer_arg > 0
    outer_arg = np.clip(outer_arg, eps, None)

    Y_t = np.log(outer_arg)
    data['Y_t'] = Y_t

    computed_values.append({
        'Year': year_i,
        'alpha': alpha_t,
        'beta': beta_t,
        'delta0': delta0,
        'delta1': delta1,
        'delta2': delta2,
        'delta3': delta3,
        'delta0_err': delta0_err,
        'delta1_err': delta1_err,
        'delta2_err': delta2_err,
        'delta3_err': delta3_err,
        'a0': a0,
        'a1': a1,
        'a2': a2,
        'a0_err': a0_err,
        'a1_err': a1_err,
        'a2_err': a2_err,
        'Y_t': Y_t,
        'C_t': Ct
    })

    seasonal_model = seasonalModel.SeasonalModel(orders=[1], periods=[365])
    seasonal_model.fit(data=data[['Y_t', 'n']], target_col='Y_t', time_col='n', include_intercept=True)

    # 6. Extract seasonal parameters and update the dictionary all at once
    a0, a1, a2 = seasonal_model._model.params.values[:3]
    a0_err, a1_err, a2_err = seasonal_model._std_errors.values[:3]
    
    model_coefficients.update({
        'Year': year_i,
        'alpha': alpha_t,
        'beta': beta_t,
        'delta0': delta0,
        'delta1': delta1,
        'delta2': delta2,
        'delta3': delta3,
        'delta0_err': delta0_err,
        'delta1_err': delta1_err,
        'delta2_err': delta2_err,
        'delta3_err': delta3_err,
        'a0': a0,
        'a1': a1,
        'a2': a2,
        'a0_err': a0_err,
        'a1_err': a1_err,
        'a2_err': a2_err
    })
    
    all_params.append(model_coefficients)
    
    # ==========================================
    # Plotting
    # ==========================================
    if plot_data:
        # Filter data between dates
        df_plot = data

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Plot 1: CAMS vs GHI
        axes[0].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[0].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[0].set_xlabel('Day of the year')
        axes[0].set_ylabel('Clear sky')
        axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[0].grid(True, linestyle='--', alpha=0.7)

        # Plot 2: Fitted vs CAMS
        axes[1].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[1].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[1].set_xlabel('Day of the year')
        axes[1].set_ylabel('Clear sky')
        axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[1].grid(True, linestyle='--', alpha=0.7)

        # Plot 3: Fitted vs GHI
        axes[2].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[2].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[2].set_xlabel('Day of the year')
        axes[2].set_ylabel('Clear sky')
        axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[2].grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()

    # ==========================================
    # Test: imputed outliers
    # ==========================================
    if test_outputs:
        # Impute outliers
        outliers = seasonalClearsky.clearsky_outliers(data[target_col], data['Ct'], data['date'], quiet=True)
        data[target_col] = outliers['x']
        
        # Test tolerance parameter
        print("\033[1;35m---------------\033[0m \033[1;32m  Test clearskyModel_control and clearskyModel_fit \033[1;35m---------------\033[0m")
        
        passed_ntol = outliers['n'] <= control['ntol']
        msg_ntol = "\033[1;32mPassed\033[0m!\n" if passed_ntol else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the number of outliers imputed is below {control['ntol']}...({outliers['n']}) {msg_ntol}")
        
        # ==========================================
        # Test: delta parameter
        # ==========================================
        
        # Test delta parameter (Accessing mangled private attribute)
        delta = clearsky_model.delta    
        test_delta = (delta > control['lower']) and (delta < control['upper'])
        msg_delta = "\033[1;32mPassed\033[0m!\n" if test_delta else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the parameter delta is inside lower ({control['lower']}) and upper ({control['upper']})...({delta}) {msg_delta}")
        
        # ==========================================
        # Test: order of seasonal components
        # ==========================================
        
        # Count the number of parameters 
        n_params_target = 1 if control['include_intercept'] else 0
        n_params_target += 1 if control['include_trend'] else 0
        n_params_target += control['orders'] * 2
        n_params_target += control['order_H0']
        
        # Test if the number of parameters is correct 
        n_params = len(clearsky_model._model.params)
        
        # Print result 
        msg_params = "\033[1;32mPassed\033[0m!\n" if n_params == n_params_target else "\033[1;31mNOT passed\033[0m! \n"
        print(f"Check if the number of parameters is equal to {n_params_target}...({n_params}) {msg_params}")
        
        # ==========================================
        # Differential
        # ==========================================
        
        # Note differential when a trend is true is not implemented
        dt = 0.05
        n0 = 34

        num_diff = (clearsky_model.predict(n=n0 + dt) - clearsky_model.predict(n=n0)) / dt
        ana_diff = clearsky_model.differential(n=n0)
        
        print(f"Numerical Differential: \n{num_diff}")
        print(f"Analytical Differential: \n{ana_diff}")


Running model with data up to 2013-12-31 (3287 rows)

Running model with data up to 2014-12-31 (3652 rows)

Running model with data up to 2015-12-31 (4017 rows)

Running model with data up to 2016-12-31 (4383 rows)

Running model with data up to 2017-12-31 (4748 rows)

Running model with data up to 2018-12-31 (5113 rows)

Running model with data up to 2019-12-31 (5478 rows)

Running model with data up to 2020-12-31 (5844 rows)

Running model with data up to 2021-12-31 (6209 rows)

Running model with data up to 2022-12-31 (6574 rows)


In [ ]:
params_df = pd.DataFrame(all_params)

# 6. Reorder the columns so 'Year' is the very first column
cols = ['Year'] + [col for col in params_df.columns if col != 'Year']
params_df = params_df[cols]
#params_df.to_csv('../results/TableA1.csv', index=False)

params_df

,Year,alpha,beta,delta0,delta1,delta2,delta3,delta0_err,delta1_err,delta2_err,delta3_err,a0,a1,a2,a0_err,a1_err,a2_err
0,2013,0.004111,0.917898,-1.171186,0.945920,0.011139,0.509035,0.319647,0.042880,0.031798,0.182858,-0.069200,-0.073676,-0.395907,0.016755,0.023702,0.023688
1,2014,0.006516,0.915698,-1.081238,0.935924,0.025055,0.450303,0.302052,0.040519,0.030048,0.172792,-0.077484,-0.068034,-0.391266,0.015798,0.022348,0.022335
2,2015,0.007020,0.915248,-1.311949,0.967261,0.000488,0.585260,0.285686,0.038324,0.028420,0.163429,-0.068364,-0.060665,-0.379721,0.015010,0.021232,0.021221
3,2016,0.005872,0.916304,-1.237456,0.956775,0.006170,0.541002,0.275402,0.036945,0.027397,0.157548,-0.071268,-0.066128,-0.377873,0.014392,0.020360,0.020346
4,2017,0.006822,0.915414,-1.156001,0.945646,0.019046,0.496073,0.264013,0.035417,0.026264,0.151032,-0.051085,-0.063663,-0.374041,0.013754,0.019457,0.019444
5,2018,0.006181,0.915987,-1.126934,0.940641,0.022330,0.483500,0.255501,0.034275,0.025417,0.146163,-0.053003,-0.076517,-0.381181,0.013255,0.018750,0.018739
6,2019,0.007463,0.914848,-1.297774,0.963266,0.012548,0.587654,0.247172,0.033157,0.024588,0.141397,-0.040697,-0.072980,-0.367049,0.012803,0.018111,0.018101
7,2020,0.008473,0.913956,-1.321108,0.967378,0.011603,0.601106,0.238579,0.032005,0.023734,0.136483,-0.029018,-0.064113,-0.359314,0.012352,0.017475,0.017463
8,2021,0.008582,0.914030,-1.494237,0.991672,-0.000809,0.700363,0.232113,0.031137,0.023091,0.132784,-0.029244,-0.059103,-0.357075,0.011897,0.016831,0.016820
9,2022,0.008348,0.914311,-1.662439,1.014590,-0.016789,0.795806,0.226603,0.030398,0.022542,0.129631,-0.022142,-0.053265,-0.354409,0.011512,0.016285,0.016275


In [43]:
params = params_df

# 1. Format the 'Train years' column
params['Train years'] = '2005-' + params['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
params['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
params['a_0_fmt'] = params.apply(lambda row: f"{row['a0']:.4f}<br>({row['a0_err']:.4f})", axis=1)
params['a_1_fmt'] = params.apply(lambda row: f"{row['a1']:.4f}<br>({row['a1_err']:.4f})", axis=1)
params['a_2_fmt'] = params.apply(lambda row: f"{row['a2']:.4f}<br>({row['a2_err']:.4f})", axis=1)

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
params['delta_0_fmt'] = params.apply(lambda row: f"{row['delta0']:.4f}<br>({row['delta0_err']:.4f})", axis=1)
params['delta_1_fmt'] = params.apply(lambda row: f"{row['delta1']:.4f}<br>({row['delta1_err']:.4f})", axis=1)
params['delta_2_fmt'] = params.apply(lambda row: f"{row['delta2']:.4f}<br>({row['delta2_err']:.4f})", axis=1)
params['delta_3_fmt'] = params.apply(lambda row: f"{row['delta3']:.4f}<br>({row['delta3_err']:.4f})", axis=1)

# Format the remaining columns to standard decimal lengths
params['alpha_fmt'] = params['alpha'].apply(lambda x: f"{x:.6f}")
params['beta_fmt'] = params['beta'].apply(lambda x: f"{x:.3f}")

# 4. Select and rename columns for the final display
display_df = params[['Train years', 'Obs.', 'alpha_fmt', 'beta_fmt', 
                     'delta_0_fmt', 'delta_1_fmt', 'delta_2_fmt', 'delta_3_fmt', 
                     'a_0_fmt', 'a_1_fmt', 'a_2_fmt']]

display_df.columns = ['Train years', 'Obs.', r'$\alpha$', r'$\beta$', 
                      r'$\delta_0$', r'$\delta_1$', r'$\delta_2$', r'$\delta_3$', 
                      r'$a_0$', r'$a_1$', r'$a_2$']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(5), td:nth-child(5)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(9), td:nth-child(9)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
styled_table


Train years,Obs.,$\alpha$,$\beta$,$\delta_0$,$\delta_1$,$\delta_2$,$\delta_3$,$a_0$,$a_1$,$a_2$
2005-2013,3287,0.004111,0.918,-1.1712(0.3196),0.9459(0.0429),0.0111(0.0318),0.5090(0.1829),-0.0692(0.0168),-0.0737(0.0237),-0.3959(0.0237)
2005-2014,3652,0.006516,0.916,-1.0812(0.3021),0.9359(0.0405),0.0251(0.0300),0.4503(0.1728),-0.0775(0.0158),-0.0680(0.0223),-0.3913(0.0223)
2005-2015,4017,0.007020,0.915,-1.3119(0.2857),0.9673(0.0383),0.0005(0.0284),0.5853(0.1634),-0.0684(0.0150),-0.0607(0.0212),-0.3797(0.0212)
2005-2016,4383,0.005872,0.916,-1.2375(0.2754),0.9568(0.0369),0.0062(0.0274),0.5410(0.1575),-0.0713(0.0144),-0.0661(0.0204),-0.3779(0.0203)
2005-2017,4748,0.006822,0.915,-1.1560(0.2640),0.9456(0.0354),0.0190(0.0263),0.4961(0.1510),-0.0511(0.0138),-0.0637(0.0195),-0.3740(0.0194)
2005-2018,5113,0.006181,0.916,-1.1269(0.2555),0.9406(0.0343),0.0223(0.0254),0.4835(0.1462),-0.0530(0.0133),-0.0765(0.0188),-0.3812(0.0187)
2005-2019,5478,0.007463,0.915,-1.2978(0.2472),0.9633(0.0332),0.0125(0.0246),0.5877(0.1414),-0.0407(0.0128),-0.0730(0.0181),-0.3670(0.0181)
2005-2020,5844,0.008473,0.914,-1.3211(0.2386),0.9674(0.0320),0.0116(0.0237),0.6011(0.1365),-0.0290(0.0124),-0.0641(0.0175),-0.3593(0.0175)
2005-2021,6209,0.008582,0.914,-1.4942(0.2321),0.9917(0.0311),-0.0008(0.0231),0.7004(0.1328),-0.0292(0.0119),-0.0591(0.0168),-0.3571(0.0168)
2005-2022,6574,0.008348,0.914,-1.6624(0.2266),1.0146(0.0304),-0.0168(0.0225),0.7958(0.1296),-0.0221(0.0115),-0.0533(0.0163),-0.3544(0.0163)


In [44]:
computed_values_df = pd.DataFrame(computed_values)
computed_values_df.to_csv('../results/TableA1_Ct_Yt.csv', index=False)

computed_values_df

,Year,alpha,beta,delta0,delta1,delta2,delta3,delta0_err,delta1_err,delta2_err,delta3_err,a0,a1,a2,a0_err,a1_err,a2_err,Y_t,C_t
0,2013,0.004111,0.917898,-1.171186,0.945920,0.011139,0.509035,0.319647,0.042880,0.031798,0.182858,-0.022142,-0.053265,-0.354409,0.011512,0.016285,0.016275,0 1.006292 1 0.702917 2 0.56...,0 2.263143 1 2.276054 2 2.29...
1,2014,0.006516,0.915698,-1.081238,0.935924,0.025055,0.450303,0.302052,0.040519,0.030048,0.172792,-0.069200,-0.073676,-0.395907,0.016755,0.023702,0.023688,0 1.018952 1 0.710494 2 0.56...,0 2.263695 1 2.276735 2 2.29...
2,2015,0.007020,0.915248,-1.311949,0.967261,0.000488,0.585260,0.285686,0.038324,0.028420,0.163429,-0.077484,-0.068034,-0.391266,0.015798,0.022348,0.022335,0 1.020109 1 0.711294 2 0.56...,0 2.264402 1 2.277388 2 2.29...
3,2016,0.005872,0.916304,-1.237456,0.956775,0.006170,0.541002,0.275402,0.036945,0.027397,0.157548,-0.068364,-0.060665,-0.379721,0.015010,0.021232,0.021221,0 1.018728 1 0.710474 2 0.56...,0 2.262316 1 2.275276 2 2.28...
4,2017,0.006822,0.915414,-1.156001,0.945646,0.019046,0.496073,0.264013,0.035417,0.026264,0.151032,-0.071268,-0.066128,-0.377873,0.014392,0.020360,0.020346,0 1.018266 1 0.710069 2 0.56...,0 2.264656 1 2.277705 2 2.29...
5,2018,0.006181,0.915987,-1.126934,0.940641,0.022330,0.483500,0.255501,0.034275,0.025417,0.146163,-0.051085,-0.063663,-0.374041,0.013754,0.019457,0.019444,0 1.011720 1 0.706078 2 0.56...,0 2.265728 1 2.278771 2 2.29...
6,2019,0.007463,0.914848,-1.297774,0.963266,0.012548,0.587654,0.247172,0.033157,0.024588,0.141397,-0.053003,-0.076517,-0.381181,0.013255,0.018750,0.018739,0 1.011292 1 0.705707 2 0.56...,0 2.268828 1 2.281966 2 2.29...
7,2020,0.008473,0.913956,-1.321108,0.967378,0.011603,0.601106,0.238579,0.032005,0.023734,0.136483,-0.040697,-0.072980,-0.367049,0.012803,0.018111,0.018101,0 1.010024 1 0.704911 2 0.56...,0 2.271643 1 2.284815 2 2.29...
8,2021,0.008582,0.914030,-1.494237,0.991672,-0.000809,0.700363,0.232113,0.031137,0.023091,0.132784,-0.029018,-0.064113,-0.359314,0.012352,0.017475,0.017463,0 1.008121 1 0.703684 2 0.56...,0 2.272668 1 2.285915 2 2.30...
9,2022,0.008348,0.914311,-1.662439,1.014590,-0.016789,0.795806,0.226603,0.030398,0.022542,0.129631,-0.029244,-0.059103,-0.357075,0.011897,0.016831,0.016820,0 1.012321 1 0.706272 2 0.56...,0 2.270493 1 2.283736 2 2.29...
